### HunyuanOCR

In [1]:
!pip install vllm>=0.12.0 pdf2image -q
!apt-get install poppler-utils -q

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [2]:
from vllm import LLM, SamplingParams
from PIL import Image
from transformers import AutoProcessor
from pdf2image import convert_from_path

In [3]:
def clean_repeated_substrings(text):
    n = len(text)
    if n < 8000:
        return text
    for length in range(2, n // 10 + 1):
        candidate = text[-length:]
        count = 0
        i = n - length
        while i >= 0 and text[i:i + length] == candidate:
            count += 1
            i -= length
        if count >= 10:
            return text[:n - length * (count - 1)]
    return text

model_path = "tencent/HunyuanOCR"
llm = LLM(model=model_path, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_path)
sampling_params = SamplingParams(temperature=0, max_tokens=16384)

INFO 01-07 09:23:17 [utils.py:253] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'tencent/HunyuanOCR'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

INFO 01-07 09:23:18 [config.py:373] Replacing legacy 'type' key with 'rope_type'


preprocessor_config.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

WARNING 01-07 09:23:44 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


INFO 01-07 09:25:34 [llm.py:360] Supported tasks: ['generate']


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
# PDF-handling
def ocr_pdf(pdf_path: str, dpi: int = 200):
    pages = convert_from_path(pdf_path, dpi=dpi)
    results = []

    for i, img in enumerate(pages, 1):
        print(f"Sida {i}/{len(pages)}...")

        messages = [
            {"role": "system", "content": ""},
            {"role": "user", "content": [
                {"type": "image", "image": "placeholder"},
                {"type": "text", "text": "Extract all information from the main body of the document image and represent it in markdown format, ignoring headers and footers. Tables should be expressed in HTML format, formulas in the document should be represented using LaTeX format, and the parsing should be organized according to the reading order."}
            ]}
        ]

        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = {"prompt": prompt, "multi_modal_data": {"image": [img]}}
        output = llm.generate([inputs], sampling_params)[0]
        results.append({"page": i, "text": clean_repeated_substrings(output.outputs[0].text)})

    return results


In [ ]:
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
    pdf_paths = [f for f in uploaded.keys() if f.endswith(".pdf")]
except ImportError:
    # Local: read from folder
    folder = Path("data/benchmark")
    pdf_paths = [str(f) for f in sorted(folder.glob("*.pdf"))]
    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs found in {folder}")

print(f"Found {len(pdf_paths)} PDF(s)")

all_results = {}

for pdf_path in pdf_paths:
    print(f"\nProcessing: {pdf_path}")
    results = ocr_pdf(pdf_path)
    full_text = "\n\n".join([r["text"] for r in results])
    all_results[pdf_path] = full_text
    print(f"Done: {len(full_text):,} chars")

Saving FlexQube.pdf to FlexQube.pdf
Sida 1/68...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Sida 2/68...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Sida 68/68...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [8]:
   # Save
output_path = "hunyuanocr_output.md"
with open(output_path, "w", encoding="utf-8") as f:
            f.write(full_text)

In [10]:
      # Preview
print(f"Preview:")
print("="*50)
print(full_text[:6000])

Preview:
# Årsredovisning 2022

## FLEXQUBE

### Cockpit

Mission Planner

Wait 8 Seconds

Go to Station Process 1

7

# Det här är FlexQube

FlexQube är en global leverantör av modulära och robusta mekaniska vagnar och robotiserade lösningar för materialhantering. Koncernen grundades 2010 och har sedan dess säkrat ett stort antal framstående företag som kunder.

FlexQube är ett teknikbolag med huvudkontor i Göteborg samt egna verksamheter i USA, Mexiko, Tyskland och England. Bolaget är verksamt inom vagnsbaserad materialhantering genom ett patenterat modulkoncept. FlexQube utvecklar och designar kundanpassade lösningar för både robotiserad och mekaniserad vagnslogistik. Genom företagets egenutvecklade och unika automationskoncept erbjuds robusta och självkörande robotvagnar. FlexQube har över 1000 kunder i 37 länder där de primära marknaderna är Nordamerika och Europa.

FlexQubes kunder återfinns inom bland annat tillverkningsindustrin, distribution- och lagerverksamheter. Exempel på 